# mini-gpt — Colab T4 Training

**Usage**: Open in VSCode with Google.colab extension connected to T4 runtime.  
Run cells top-to-bottom on first setup. For resume after session reset, skip to Cell 9.

---

In [ ]:

# Cell 1: GPU check
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

: 

In [ ]:
# Cell 2: Install dependencies
!pip install tokenizers wandb -q

In [ ]:
# Cell 3: Clone repo (or pull latest if already cloned)
import os
if not os.path.exists('/content/mini-gpt'):
    !git clone https://github.com/daisukeabe32/mini-gpt /content/mini-gpt -q
    print("cloned")
else:
    !git -C /content/mini-gpt pull -q
    print("pulled latest")
%cd /content/mini-gpt
!git log --oneline -3

In [ ]:
# Cell 4: Mount Google Drive
# If this fails via the extension, run it in the Colab browser tab instead.
# After mounting in either place, /content/drive/MyDrive/ is accessible here.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 5: Copy tokenized data Drive -> local (once per session, ~2-3 min)
import os, shutil
src = "/content/drive/MyDrive/mini-gpt-data/bpe_hf_30000_tinystories"
dst = "data/tokenized/bpe_hf_30000_tinystories"
os.makedirs(dst, exist_ok=True)
for fname in ["tokenizer.json", "hf_tokenizer.json", "val_ids.pt", "train_ids.pt"]:
    dst_path = f"{dst}/{fname}"
    if not os.path.exists(dst_path):
        print(f"copying {fname} ...")
        shutil.copy2(f"{src}/{fname}", dst_path)
        print(f"  done ({os.path.getsize(dst_path)/1e6:.0f} MB)")
    else:
        print(f"  {fname} — skip (exists)")

In [ ]:
# Cell 6: Verify tokenized data
import torch
train = torch.load("data/tokenized/bpe_hf_30000_tinystories/train_ids.pt")
val   = torch.load("data/tokenized/bpe_hf_30000_tinystories/val_ids.pt")
print(f"train: {len(train):,} tokens")
print(f"val  : {len(val):,} tokens")

In [ ]:
# Cell 7: W&B login
# API key: /Users/daison32/Documents/Claude/Projects/スイスMaster/api_keys.txt
import wandb
wandb.login()

In [ ]:
# Cell 8: START TRAINING — fresh run (block_size=256, d_model=256)
# Checkpoints saved to Drive every 5000 steps.
import os
CKPT_DIR = "/content/drive/MyDrive/mini-gpt-checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

!python -u -m scripts.train_char_gpt \
    --tokenized_dir  data/tokenized/bpe_hf_30000_tinystories \
    --block_size  256 \
    --d_model     256 \
    --n_layers      6 \
    --num_heads     8 \
    --d_k          32 \
    --d_ff       1024 \
    --batch_size   32 \
    --max_iters  80000 \
    --lr          3e-4 \
    --min_lr      3e-5 \
    --warmup_iters 500 \
    --eval_every  2500 \
    --save_every  5000 \
    --checkpoint_dir {CKPT_DIR} \
    2>&1 | tee {CKPT_DIR}/training_log.txt

In [ ]:
# Cell 9: RESUME TRAINING — after session reset
# 1. Check Drive for the run_id: /content/drive/MyDrive/mini-gpt-checkpoints/<run_id>/
# 2. Replace RUN_ID below
import os
CKPT_DIR = "/content/drive/MyDrive/mini-gpt-checkpoints"

# Show available runs to find RUN_ID
print("Available runs:")
for d in sorted(os.listdir(CKPT_DIR)):
    if os.path.isdir(f"{CKPT_DIR}/{d}"):
        files = os.listdir(f"{CKPT_DIR}/{d}")
        print(f"  {d}  {files}")

In [ ]:
# Cell 10: RESUME — set RUN_ID from Cell 9 output, then run
import os
CKPT_DIR = "/content/drive/MyDrive/mini-gpt-checkpoints"
RUN_ID    = "XXXXXXXX_XXXXXX"  # <-- replace with actual run_id from Cell 9
RESUME_PT = f"{CKPT_DIR}/{RUN_ID}/best.pt"
print(f"Resuming from: {RESUME_PT}")
print(f"Exists: {os.path.exists(RESUME_PT)}")

!python -u -m scripts.train_char_gpt \
    --tokenized_dir  data/tokenized/bpe_hf_30000_tinystories \
    --block_size  256 \
    --d_model     256 \
    --n_layers      6 \
    --num_heads     8 \
    --d_k          32 \
    --d_ff       1024 \
    --batch_size   32 \
    --max_iters  80000 \
    --lr          3e-4 \
    --min_lr      3e-5 \
    --warmup_iters 500 \
    --eval_every  2500 \
    --save_every  5000 \
    --checkpoint_dir {CKPT_DIR} \
    --resume_from {RESUME_PT} \
    2>&1 | tee {CKPT_DIR}/training_log_resumed.txt

In [ ]:
# Cell 11: Check progress
import os
CKPT_DIR = "/content/drive/MyDrive/mini-gpt-checkpoints"
for log_name in ["training_log.txt", "training_log_resumed.txt"]:
    log_path = f"{CKPT_DIR}/{log_name}"
    if not os.path.exists(log_path):
        continue
    with open(log_path) as f:
        lines = [l for l in f if l.startswith("step")]
    print(f"=== {log_name} ({len(lines)} evals) ===")
    if lines:
        print("first :", lines[0].strip())
        print("latest:", lines[-1].strip())

In [ ]:
# Cell 12: Induction head analysis (run after training)
# Replace RUN_ID with actual value
import os
CKPT_DIR = "/content/drive/MyDrive/mini-gpt-checkpoints"
RUN_ID   = "XXXXXXXX_XXXXXX"  # <-- replace
CKPT_PT  = f"{CKPT_DIR}/{RUN_ID}/best.pt"
os.makedirs("figs", exist_ok=True)

!python -m scripts.analyze_attention \
    --checkpoint {CKPT_PT} \
    --seq_len 64 \
    --out_dir figs/